# Introduce Featurization Module to FLAML
In our latest version of FLAML, we introduce the `featurization` module, which could perform automatically search for feature engineering pipeline.
Currently available search stages are: feature selection, feature transform for categorical and numerical features.

## Install latest FLAML

## Configure Experiment

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=ConvergenceWarning)
seed = 3407
from flaml import AutoML
import time

## Load Data
We use Volkert dataset in this demo. The Volkert datasetis a significant resource in recommender system research, derived from a German online shop's transactional data. It features anonymized information like user IDs, product IDs, and purchase timestamps. Key statistical attributes of this dataset include:

- #lines     : 58310
- #features  : 181
- #classes   : 10

The difficulty is that there is a broad diversity of data types and distributions (including balanced or unbalanced classes, sparse or dense feature representations, with or without missing values or categorical variables, various metrics of evaluation, various proportions of number of features and number of examples). The problems are drawn from a wide variety of domains and include medical diagnosis from laboratory analyses, speech recognition, credit rating, prediction or drug toxicity or efficacy, classification of text, prediction of customer satisfaction, object recognition, protein structure prediction, action recognition in video data, etc. While there exist machine learning toolkits including methods that can solve all these problems, it is still considerable human effort to find, for a given combination of dataset, task, metric of evaluation, and available computational time, the combination of methods and hyper-parameter setting that is best suited. Your challenge is to create the "perfect black box" eliminating the human in the loop. This dataset is widely used for testing various recommendation algorithms and serves as a benchmark for comparing different techniques in the field.

In [ ]:
import pandas as pd
import openml
from sklearn import datasets
from sklearn.model_selection import train_test_split
dataset = openml.datasets.get_dataset(41166)

df, y, categorical_indicator, attribute_names = dataset.get_data(dataset_format="dataframe", target=dataset.default_target_attribute)
df['target'] = y
train, test = train_test_split(df, test_size=0.2, random_state=seed, shuffle=True)
display(df)

In [ ]:
automl_settings = {
    "task": "classification",
    "label": "target",
    "metric": "accuracy",
    "seed": seed,
    "log_file_name": "featurization.log",
    "log_type": "all",
    "max_iter": 10,
    "time_budget": 600,
    "estimator_list":  ['sgd'],
}

## AutoML on Raw Data
First, we perform AutoML on raw dataset, to provide a baseline of the experiment.

In [ ]:
automl = AutoML()
automl.fit(
    dataframe=train, 
    featurization="off",
    **automl_settings,
)

In [ ]:
from sklearn.metrics import r2_score, accuracy_score
pred = automl.predict(test)
if automl_settings['task'] == 'classification':
    score = accuracy_score(test[automl_settings['label']], pred)
    print("Accuracy without Featurization: {}".format(score))
else:
    score = r2_score(test[automl_settings['label']], pred)
    print("R2 score without Featurization: {}".format(score))

## AutoML with Featurization
Next, we introduce the latest `featurization` module, which could automatically search for a feature engineering pipeline along with AutoML process.

This module leverages HPO algorithms to intelligently select, transform, and construct features from raw data, enhancing the model's predictive power. 

The module's integration with AutoML allows for a seamless, automated process where both feature engineering and model selection are jointly optimized.

Set the `featurization` parameter to `auto` to tune feature engineering, or to `force` to choose a non-null method for each supported stage. The default is `off`, including on Fabric. Enable it explicitly with `auto` or `force`, or set the `FLAML_FEATURIZATION` environment variable. An explicit parameter overrides the environment default.



Currently avaliable feature engineering methods:
1. Stage `categorical`: Methods to encode categorical features. Available:
  - `ordinal`: Ordinal encoding for categorical features.
  

2. Stage `numerical`: Methods to transform numerical features. Available:
  - `null`: No transformation applied to numerical features.
  - `scaler_standard`: Standard scaling for numerical features, normalizing them to have zero mean and unit variance.
  - `scaler_minmax`: Min-Max scaling, transforming features by scaling each feature to a given range, typically [0, 1].
  - `scaler_maxabs`: MaxAbs scaling, scales each feature by its maximum absolute value. This is meant for data that is already centered at zero or sparse data.
  - `scaler_robust`: Robust scaling using statistics that are robust to outliers, particularly useful when dealing with features that contain many outliers.
  - `normalizer_sparse`: Normalization applied to sparse input, making each feature vector have unit norm.
    
  
3. Stage `selection`: Methods for feature selection. Available:
  - `null`: No feature selection is applied.
  - `cardinality`: Selecting features based on their cardinality.
  - `variance`: Selecting features based on variance threshold.
  

4. Stage `extraction`: Feature extraction methods, applicable based on task type. Available:
  - `null`: No feature extraction is applied.
  - `PCA`: Principal Component Analysis.
  - `LDA`: Linear Discriminant Analysis(For classification tasks only).



In [ ]:
automl = AutoML()
automl.fit(
    dataframe=train, 
    featurization="auto",
    **automl_settings,
)

In [ ]:
from sklearn.metrics import r2_score, accuracy_score
pred = automl.predict(test)
if automl_settings['task'] == 'classification':
    score = accuracy_score(test[automl_settings['label']], pred)
    print("Accuracy with Featurization: {}".format(score))
else:
    score = r2_score(test[automl_settings['label']], pred)
    print("R2 score with Featurization: {}".format(score))

### Standalone Feturization Pipeline 
Once the AutoML process completes, the featurization pipeline can be accessed independently, and be utilized separately from the AutoML process.

You can retrieve the feature engineering pipeline specifically through `automl.model.autofe`. 
Alternatively, for a comprehensive view of the preprocessing steps, including those from FLAML's existing preprocessors, use `automl.feature_transformer`.

- To view the configuration details of the entire pipeline, use `autofe.show_transformations()`.
- For a more interactive experience, the pipeline structure can be visualized by executing `display(autofe)` or simply `autofe`.

In [ ]:
autofe = automl.model.autofe
display(autofe)
# The lower-level featurizer takes task-preprocessed features without target labels.
X_test = test.drop(columns=["target"])
display(autofe.transform(automl.preprocess(X_test)))

Full data preprocessor set, including FLAML's existing preprocess and Featurization:

In [ ]:
transformer = automl.feature_transformer
display(transformer)
display(transformer.transform(test))

In [ ]:
autofe.show_transformations()

#### Export Featurization as config and Reconstruct using config
You can access `autofe.config` to retrieve the config printed above as a list. 

You can save and reload this config to reconstruct the Featurization pipeline

In [ ]:
import json
config_to_save = autofe.config
with open('autofe_config.json', 'w') as f:
    json.dump(config_to_save, f)

In [ ]:
from flaml.automl import Featurization
reload_config = json.load(open('autofe_config.json', 'r'))
re_autofe = Featurization(config=reload_config, task=automl_settings['task'])
X = train.drop(columns=['target'])
y = train['target']
re_autofe.fit(X, y)
display(re_autofe)
display(re_autofe.transform(test.drop(columns=["target"])))

### MLflow Model Registry and Deploy
We offer convinient entry for user to register the best AutoML pipeline(contains FLAML preprocessor, Featurization and Estimator) in MLflow.

Afterward, user could easily deploy the model or fetch and load the model.

In [ ]:
import flaml
model_version = flaml.automl.register_automl_pipeline(automl, model_name="automl_pipeline_Volkert")

Perform prediction with registered models with `scikit-learn`.

In [ ]:
import mlflow
loaded_pipeline = mlflow.sklearn.load_model(model_uri=f"models:/{model_version.name}/{model_version.version}")
display(loaded_pipeline)
loaded_pipeline.predict(test)

### Featurization Process Analysis
Using FLAML's visulization module, we could see the relationship between performance and each step of featurization in these slice plot. 

We could see that featurization run has generally higher score (score = 1 - validation loss) than run without any featurization method (`null`). 

In [ ]:
# # uncomment and try different renderers if the figure is not shown in the next cell
# import plotly.io as pio
# pio.renderers.default = "notebook"  # "notebook", "sphinx_gallery"

In [ ]:
import flaml.visualization as fviz
fig = fviz.plot_slice(automl, params=['fe.numerical', 'fe.extraction'])
fig.show()